# KIM form probe -- SA²'s two-sided, signed second difference vs KIM's form

Rule: `core/docs/final/KIM_RUN.md` §14, written before any number. **Inference only on K0 s2024, nothing is trained or extracted, K5 stays the main model.** One seed: **PRELIMINARY (n = 1)**. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K0, its reads, the pilot, the bakes) | no | 2-3 min |
| 4 | tests on the probe -- **raises if red** | no | 1-2 min |
| 5 | the probe: stops if `vt2` / `clip2` / `host` do not reproduce the recorded pilot | **yes** | 10-20 min |
| 6 | **paste `REPORTS/form_probe/kim_form_probe.md` to Claude** | no | seconds |

A re-run with the report on Drive costs seconds (C39). Thesis and Thesis-V2 are only read; the report goes to Thesis-Final.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 bakes
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K0, its reads, the pilot; the report goes here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/kim_form_probe.py').is_file(), 'this upload has no kim_form_probe -- re-upload kat-vad/'
assert '## 14. Form probe' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §14 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K0       = constants.KIM_CONTROL                   # the pilot's host (KIM_RUN.md §14)
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
V2_CACHE = Path(DRIVE_V2) / 'cache'
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{constants.TAD_DATASET}_s8_nexar'
T2       = constants.DATA_ROOT / T2NAME
TAD      = constants.DATA_ROOT / constants.TAD_DATASET
DOTA     = constants.DATA_ROOT / 'DoTA'
ROOT     = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME    # Thesis-Final/outputs/kim
REPORTS  = ROOT / 'REPORTS'
CKPT     = ROOT / K0 / f's{SEED}' / 'checkpoint_last.pt'
BASELINE = REPORTS / 'pilot_edges' / 'kim_pilot.json'
EVAL     = {s: ROOT / 'eval' / s / K0 for s in ('dota', 'dada', 'tad')}
OUT      = REPORTS / constants.KIM_FORM_REPORT_DIRNAME
REPORT   = OUT / 'kim_form_probe.md'
LOCAL    = Path('/content/kim_form')
LOCAL.mkdir(parents=True, exist_ok=True)

for name, path in (('K0 checkpoint', CKPT), ('recorded pilot', BASELINE), ('K0 DoTA read', EVAL['dota']),
                   ('K0 DADA read', EVAL['dada']), ('K0 TAD read', EVAL['tad'] / f's{SEED}' / 'scores'),
                   ('DoTA-CAP A2 bake', A2_DOTA), ('DADA A2 bake', A2_DADA), ('TAD A2 bake', A2_TAD),
                   ('DoTA metadata', DOTA / 'metadata_val.json'), ('DoTA split', DOTA / 'val_split.txt'),
                   ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:18s} {path}')
pilot = json.loads(BASELINE.read_text())
assert pilot['host'] == K0, f"the recorded pilot read {pilot['host']}, §14 reads {K0}"
print(f"recorded pilot: {pilot['host']} step {pilot['global_step']} | report -> {OUT}")

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 5 needs one'

In [ ]:
import torch

from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370, len(DADA_IDS)
assert len(TAD_IDS) == constants.V2_TADZ_VIDEOS, len(TAD_IDS)
for name, d, ids in (('DADA', A2_DADA, DADA_IDS), ('DoTA-CAP', A2_DOTA, CAP_IDS), ('TAD', A2_TAD, TAD_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} {name} videos have no A2 bake: {miss[:5]}'
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
print(f'preflight OK: {len(DADA_IDS)} DADA, {len(CAP_IDS)} DoTA-CAP-dev, {len(TAD_IDS)} TAD videos')

## 4. Tests on the probe (CPU) -- **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim_form_probe.py',
     'core/tests/test_kim_tools.py'])

## 5. The probe (GPU, inference only)

It raises, and writes nothing, if `vt2`, `clip2` or `host` do not reproduce the recorded pilot: paste the error to Claude and do not read any number.

In [ ]:
if REPORT.exists():
    print(f'report already on Drive, nothing to do: {REPORT}')
else:
    assert torch.cuda.is_available(), 'torch sees no GPU'
    host = {s: stage(EVAL[s], LOCAL / 'host_read' / s) for s in ('dota', 'dada')}
    stage(EVAL['tad'] / f's{SEED}' / 'scores', LOCAL / 'host_read' / 'tad' / 'scores')
    bake = {'dota': stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), 'dada': stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
            'tad': stage(A2_TAD, LOCAL / 'bake_tad', TAD_IDS)}
    local_out = LOCAL / 'form_probe'
    run([sys.executable, '-m', 'core.tools.kim_form_probe', '--name', K0, '--ckpt', CKPT, '--baseline', BASELINE,
         '--set', CAP_DEV, 'native', host['dota'], bake['dota'],
         '--set', 'dada', 'rows', host['dada'], bake['dada'],
         '--set', 'tad', 'rows', LOCAL / 'host_read' / 'tad', bake['tad'],
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--out-dir', local_out], log=LOCAL / 'form_probe.log')
    publish(local_out, OUT)
    drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
    drive.mount('/content/drive')

## 6. **Paste `REPORTS/form_probe/kim_form_probe.md` to Claude**

In [ ]:
print(REPORT.read_text())
decision = json.loads((OUT / 'kim_form_probe.json').read_text())['decision']
print(f">>> verdict {decision['verdict']} | passes {decision['passes'] or 'none'}: nothing trains from this notebook")